# 1. Setup

## Libraries and imports

In [108]:
!pip install pandas


[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [109]:
import pandas as pd
import numpy as np

## Loading NHASES datasets

In [110]:
# DEMOGRAPHIC DATA
demo = pd.read_sas("./data/DEMO_L.xpt", format="xport")

demo

,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDRETH1,RIDRETH3,RIDEXMON,RIDEXAGM,...,DMDHRGND,DMDHRAGZ,DMDHREDZ,DMDHRMAZ,DMDHSEDZ,WTINT2YR,WTMEC2YR,SDMVSTRA,SDMVPSU,INDFMPIR
0,130378.0,12.0,2.0,1.0,43.0,NaN,5.0,6.0,2.0,NaN,...,NaN,NaN,NaN,NaN,NaN,50055.450807,54374.463898,173.0,2.0,5.00
1,130379.0,12.0,2.0,1.0,66.0,NaN,3.0,3.0,2.0,NaN,...,NaN,NaN,NaN,NaN,NaN,29087.450605,34084.721548,173.0,2.0,5.00
2,130380.0,12.0,2.0,2.0,44.0,NaN,2.0,2.0,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,80062.674301,81196.277992,174.0,1.0,1.41
3,130381.0,12.0,2.0,2.0,5.0,NaN,5.0,7.0,1.0,71.0,...,2.0,2.0,2.0,3.0,NaN,38807.268902,55698.607106,182.0,2.0,1.53
4,130382.0,12.0,2.0,1.0,2.0,NaN,3.0,3.0,2.0,34.0,...,2.0,2.0,3.0,1.0,2.0,30607.519774,36434.146346,182.0,2.0,3.60
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11928,142306.0,12.0,2.0,1.0,9.0,NaN,2.0,2.0,1.0,111.0,...,1.0,3.0,3.0,3.0,NaN,11147.192563,13459.129019,176.0,1.0,2.01
11929,142307.0,12.0,2.0,2.0,49.0,NaN,4.0,4.0,2.0,NaN,...,NaN,NaN,NaN,NaN,NaN,69419.620456,64962.328962,181.0,1.0,NaN
11930,142308.0,12.0,2.0,1.0,50.0,NaN,2.0,2.0,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,32696.313477,44367.534132,183.0,2.0,1.95
11931,142309.0,12.0,2.0,1.0,40.0,NaN,2.0,2.0,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,30547.974564,46249.361849,176.0,1.0,3.11


In [111]:
# EXAMINATION DATA
bmx = pd.read_sas("./data/BMX_L.xpt", format="xport") # body measures
bpxo = pd.read_sas("./data/BPXO_L.xpt", format="xport") # measured blood pressure

In [112]:
# LABORATORY DATA
ghb = pd.read_sas("./data/GHB_L.xpt", format="xport") # glycohemoglobin (HbA1c)
glu = pd.read_sas("./data/GLU_L.xpt", format="xport") # fasting glucose (optional validation)

In [113]:
# QUESTIONNAIRE DATA
paq = pd.read_sas("./data/PAQ_L.xpt", format="xport") # physical activity and sedentary behavior
smq = pd.read_sas("./data/SMQ_L.xpt", format="xport")
bpq = pd.read_sas("./data/BPQ_L.xpt", format="xport")
diq = pd.read_sas("./data/DIQ_L.xpt", format="xport")

## Setting up audit log

In [114]:
audit_log = []

In [115]:
def add_audit_finding(
    dataset,
    column,
    issue,
    example=None,
    action_stage=None,
    planned_action=None
):
    finding = {
        "dataset": dataset,
        "column": column,
        "issue": issue,
        "example": example,
        "action_stage": action_stage,
        "planned_action": planned_action
    }

    audit_log.append(finding)

# 2. Retaining and Renaming Required Columns

## Demographic Data

In [116]:
demo.columns

Index(['SEQN', 'SDDSRVYR', 'RIDSTATR', 'RIAGENDR', 'RIDAGEYR', 'RIDAGEMN',
       'RIDRETH1', 'RIDRETH3', 'RIDEXMON', 'RIDEXAGM', 'DMQMILIZ', 'DMDBORN4',
       'DMDYRUSR', 'DMDEDUC2', 'DMDMARTZ', 'RIDEXPRG', 'DMDHHSIZ', 'DMDHRGND',
       'DMDHRAGZ', 'DMDHREDZ', 'DMDHRMAZ', 'DMDHSEDZ', 'WTINT2YR', 'WTMEC2YR',
       'SDMVSTRA', 'SDMVPSU', 'INDFMPIR'],
      dtype='str')

In [117]:
demo_req_columns = demo[[
    'SEQN', 'RIDAGEYR', 'RIAGENDR', 
    'RIDRETH3', 'DMDEDUC2', 'INDFMPIR', 
    'RIDEXPRG', 'WTMEC2YR', 'SDMVSTRA', 
    'SDMVPSU'
    ]].rename(
    columns = {
        'SEQN' : 'participant_id', 
        'RIDAGEYR' : 'age_years', 
        'RIAGENDR' : 'sex', 
        'RIDRETH3' : 'race_ethnicity', 
        'DMDEDUC2': 'education_level', 
        'INDFMPIR' : 'income_poverty_ratio', 
        'RIDEXPRG' : 'pregnancy_status', 
        'WTMEC2YR' : 'mec_exam_weight', 
        'SDMVSTRA' : 'survey_stratum', 
        'SDMVPSU' : 'survey_psu'
    }
    )

demo_req_columns

,participant_id,age_years,sex,race_ethnicity,education_level,income_poverty_ratio,pregnancy_status,mec_exam_weight,survey_stratum,survey_psu
0,130378.0,43.0,1.0,6.0,5.0,5.00,NaN,54374.463898,173.0,2.0
1,130379.0,66.0,1.0,3.0,5.0,5.00,NaN,34084.721548,173.0,2.0
2,130380.0,44.0,2.0,2.0,3.0,1.41,2.0,81196.277992,174.0,1.0
3,130381.0,5.0,2.0,7.0,NaN,1.53,NaN,55698.607106,182.0,2.0
4,130382.0,2.0,1.0,3.0,NaN,3.60,NaN,36434.146346,182.0,2.0
...,...,...,...,...,...,...,...,...,...,...
11928,142306.0,9.0,1.0,2.0,NaN,2.01,NaN,13459.129019,176.0,1.0
11929,142307.0,49.0,2.0,4.0,5.0,NaN,NaN,64962.328962,181.0,1.0
11930,142308.0,50.0,1.0,2.0,4.0,1.95,NaN,44367.534132,183.0,2.0
11931,142309.0,40.0,1.0,2.0,4.0,3.11,NaN,46249.361849,176.0,1.0


## Examination Data

In [118]:
bmx.columns
bmx_req_columns = bmx[[
    'SEQN', 'BMXBMI', 'BMXWAIST'
]].rename(
    columns = {
        'SEQN' : 'participant_id', 
        'BMXBMI' : 'bmi', 
        'BMXWAIST' : 'waist_cm'
    }
)

In [119]:
bpxo.columns
bpxo_req_columns = bpxo[[
    'SEQN', 'BPXOSY1', 'BPXOSY2', 
    'BPXOSY3', 'BPXODI1', 'BPXODI2', 
    'BPXODI3'
]].rename(
    columns = {
        'SEQN' : 'participant_id', 
        'BPXOSY1' : 'systolic_1', 
        'BPXOSY2' : 'systolic_2',
        'BPXOSY3' : 'systolic_3',
        'BPXODI1' : 'diastolic_1',
        'BPXODI2' : 'diastolic_2',
        'BPXODI3' : 'diastolic_3'
    }
)

# derived : mean_systolic_bp, mean_diastolic_bp

## Laboratory Data

In [120]:
ghb.columns

ghb_req_columns = ghb.rename(columns = {
    'SEQN' : 'participant_id', 
    'LBXGH' : 'hba1c_pct',    
    'WTPH2YR' : 'phlebotomy_weight', 

})

In [121]:
glu.columns

glu_req_columns = glu[[
    'SEQN', 'LBXGLU', 'WTSAF2YR'
]].rename( columns = {
    'SEQN' : 'participant_id', 
    'LBXGLU' : 'fasting_glucose_mg_dl', 
    'WTSAF2YR' : 'fasting_subsample_weight'
})

## Questionnaire Data

In [122]:
paq_req_columns = paq.rename(columns = {
    'SEQN' : 'participant_id', 
    'PAD790Q' : 'moderate_pa_frequency', 
    'PAD790U' : 'moderate_pa_unit', 
    'PAD800' : 'moderate_pa_minutes_session', 
    'PAD810Q' : 'vigorous_pa_frequency', 
    'PAD810U' : 'vigorous_pa_unit', 
    'PAD820' : 'vigorous_pa_minutes_session',
    'PAD680' : 'sedentary_min_day'
})

In [123]:
smq_req_columns = smq[['SEQN', 'SMQ020', 'SMQ040']].rename(columns = {
    'SEQN' : 'participant_id', 
    'SMQ020' : 'ever_100_cigarettes', 
    'SMQ040' : 'current_smoking_frequency', 
})

# derived: smoking_status

In [124]:
bpq_req_columns = bpq[['SEQN', 'BPQ020', 'BPQ150', 'BPQ080', 'BPQ101D']].rename(columns = {
    'SEQN' : 'participant_id', 
    'BPQ020' : 'hypertension_history', 
    'BPQ150' : 'bp_medication', 
    'BPQ080' : 'high_cholesterol_history', 
    'BPQ101D' : 'cholesterol_medication'
})

In [125]:
diq_req_columns = diq[[
    'SEQN', 'DIQ010', 'DIQ160'
]].rename(columns = {
    'SEQN' : 'participant_id', 
    'DIQ010' : 'diagnosed_diabetes', 
    'DIQ160' : 'prediabetes_history', 
})

# 3. Establishing Cohort
Cohort represents adults ages 20+ and excludes women that are pregnant or cannot ascertain pregnancy status

In [126]:
# xpt file read 0 as extremely small float value
# replacing this with 0

demo_req_columns = demo_req_columns.replace(
    5.397605346934028e-79,
    0
)

demo_req_columns

,participant_id,age_years,sex,race_ethnicity,education_level,income_poverty_ratio,pregnancy_status,mec_exam_weight,survey_stratum,survey_psu
0,130378.0,43.0,1.0,6.0,5.0,5.00,NaN,54374.463898,173.0,2.0
1,130379.0,66.0,1.0,3.0,5.0,5.00,NaN,34084.721548,173.0,2.0
2,130380.0,44.0,2.0,2.0,3.0,1.41,2.0,81196.277992,174.0,1.0
3,130381.0,5.0,2.0,7.0,NaN,1.53,NaN,55698.607106,182.0,2.0
4,130382.0,2.0,1.0,3.0,NaN,3.60,NaN,36434.146346,182.0,2.0
...,...,...,...,...,...,...,...,...,...,...
11928,142306.0,9.0,1.0,2.0,NaN,2.01,NaN,13459.129019,176.0,1.0
11929,142307.0,49.0,2.0,4.0,5.0,NaN,NaN,64962.328962,181.0,1.0
11930,142308.0,50.0,1.0,2.0,4.0,1.95,NaN,44367.534132,183.0,2.0
11931,142309.0,40.0,1.0,2.0,4.0,3.11,NaN,46249.361849,176.0,1.0


In [127]:
# normalizing education_level so 7 and 9 are treated as NaN (missing value)
# Adults 20+: 
# 1 <9th grade; 
# 2 9-11th; 
# 3 HS/GED; 
# 4 some college/AA; 
# 5 college+. 
# Codes 7/9 -> missing.

demo_req_columns['education_level'] = demo_req_columns['education_level'].replace([7,9], np.nan)
demo_req_columns['education_level'].unique()

array([ 5.,  3., nan,  2.,  4.,  1.])

In [128]:
# creating adult cohort 
# age_years >= 20

demo_adults = demo_req_columns[
    demo_req_columns['age_years'] >= 20
].copy()

demo_adults

,participant_id,age_years,sex,race_ethnicity,education_level,income_poverty_ratio,pregnancy_status,mec_exam_weight,survey_stratum,survey_psu
0,130378.0,43.0,1.0,6.0,5.0,5.00,NaN,54374.463898,173.0,2.0
1,130379.0,66.0,1.0,3.0,5.0,5.00,NaN,34084.721548,173.0,2.0
2,130380.0,44.0,2.0,2.0,3.0,1.41,2.0,81196.277992,174.0,1.0
6,130384.0,43.0,1.0,1.0,2.0,0.63,NaN,0.000000,179.0,2.0
7,130385.0,65.0,2.0,3.0,3.0,5.00,NaN,0.000000,187.0,2.0
...,...,...,...,...,...,...,...,...,...,...
11927,142305.0,76.0,2.0,1.0,1.0,2.25,NaN,43483.407534,180.0,2.0
11929,142307.0,49.0,2.0,4.0,5.0,NaN,NaN,64962.328962,181.0,1.0
11930,142308.0,50.0,1.0,2.0,4.0,1.95,NaN,44367.534132,183.0,2.0
11931,142309.0,40.0,1.0,2.0,4.0,3.11,NaN,46249.361849,176.0,1.0


In [129]:
demo_adults['education_level'].isna().mean()*100

# only 0.3% of adult education level is missing, so these rows can be dropped
# missing education level also have a lot of missing income poverty ratio

np.float64(0.33294916122422846)

In [130]:
# checking pregnancy status
# For females 20-44: 
# 1 pregnant; 
# 2 not pregnant; 
# 3 cannot ascertain.

# there are 41 women with a pregnancy status of 1
# there are 397 women with a pregnancy status of 3

demo_adults['pregnancy_status'].value_counts(dropna=False)

pregnancy_status
NaN    6306
2.0    1065
3.0     397
1.0      41
Name: count, dtype: int64

In [131]:
# dropping pregnancy status 1 and pregnancy status 3

demo_cohort = demo_adults[
    ~(
        (demo_adults["pregnancy_status"] == 1) |
        ((demo_adults["pregnancy_status"] == 3) & (demo_adults["sex"] == 2))
    )
].copy()

demo_cohort['pregnancy_status'].value_counts(dropna=False)

demo_cohort

,participant_id,age_years,sex,race_ethnicity,education_level,income_poverty_ratio,pregnancy_status,mec_exam_weight,survey_stratum,survey_psu
0,130378.0,43.0,1.0,6.0,5.0,5.00,NaN,54374.463898,173.0,2.0
1,130379.0,66.0,1.0,3.0,5.0,5.00,NaN,34084.721548,173.0,2.0
2,130380.0,44.0,2.0,2.0,3.0,1.41,2.0,81196.277992,174.0,1.0
6,130384.0,43.0,1.0,1.0,2.0,0.63,NaN,0.000000,179.0,2.0
7,130385.0,65.0,2.0,3.0,3.0,5.00,NaN,0.000000,187.0,2.0
...,...,...,...,...,...,...,...,...,...,...
11927,142305.0,76.0,2.0,1.0,1.0,2.25,NaN,43483.407534,180.0,2.0
11929,142307.0,49.0,2.0,4.0,5.0,NaN,NaN,64962.328962,181.0,1.0
11930,142308.0,50.0,1.0,2.0,4.0,1.95,NaN,44367.534132,183.0,2.0
11931,142309.0,40.0,1.0,2.0,4.0,3.11,NaN,46249.361849,176.0,1.0


In [132]:
# making sure there are no NaN values for females between 20 and 44

pregnancy_nan_check = demo_cohort[
    (demo_cohort["sex"] == 2) &
    (demo_cohort["age_years"].between(20, 44)) &
    (demo_cohort["pregnancy_status"].isna())
]

pregnancy_nan_check[
    ["participant_id", "age_years", "sex", "pregnancy_status"]
]

,participant_id,age_years,sex,pregnancy_status


In [133]:
# we can drop pregnancy column now since it was only useful as a cohort filter

demo_cohort = demo_cohort.drop(columns=["pregnancy_status"])
demo_cohort

# demo cohort now only consists of adults >= 20 years, and candidates that are not pregnant or cannot ascertain if they are pregnant

,participant_id,age_years,sex,race_ethnicity,education_level,income_poverty_ratio,mec_exam_weight,survey_stratum,survey_psu
0,130378.0,43.0,1.0,6.0,5.0,5.00,54374.463898,173.0,2.0
1,130379.0,66.0,1.0,3.0,5.0,5.00,34084.721548,173.0,2.0
2,130380.0,44.0,2.0,2.0,3.0,1.41,81196.277992,174.0,1.0
6,130384.0,43.0,1.0,1.0,2.0,0.63,0.000000,179.0,2.0
7,130385.0,65.0,2.0,3.0,3.0,5.00,0.000000,187.0,2.0
...,...,...,...,...,...,...,...,...,...
11927,142305.0,76.0,2.0,1.0,1.0,2.25,43483.407534,180.0,2.0
11929,142307.0,49.0,2.0,4.0,5.0,NaN,64962.328962,181.0,1.0
11930,142308.0,50.0,1.0,2.0,4.0,1.95,44367.534132,183.0,2.0
11931,142309.0,40.0,1.0,2.0,4.0,3.11,46249.361849,176.0,1.0


# 4. Auditing Data

## Componenet coverage audit

Of the 7,371 eligible adults, how many actually have a record in each NHANES component?

In [134]:
cohort_ids = demo_cohort["participant_id"]

coverage_results = []

for name, df in datasets.items():

    available = cohort_ids.isin(df["participant_id"])

    coverage_results.append({
        "dataset": name.upper(),
        "cohort_size": len(demo_cohort),
        "participants_available": available.sum(),
        "participants_not_available": (~available).sum(),
        "coverage_percent": round(available.mean() * 100, 2)
    })

coverage_summary = pd.DataFrame(coverage_results)

coverage_summary

,dataset,cohort_size,participants_available,participants_not_available,coverage_percent
0,DEMO,7371,7371,0,100.00
1,BMX,7371,5995,1376,81.33
2,PAQ,7371,7371,0,100.00
3,SMQ,7371,7371,0,100.00
4,BPXO,7371,5995,1376,81.33
5,BPQ,7371,7371,0,100.00
6,DIQ,7371,7371,0,100.00
7,GHB,7371,5995,1376,81.33
8,GLU,7371,3382,3989,45.88


## Source level Missingness
Among the people who appear in this original NHANES file, how much of this variable is missing?

In [135]:
# python read paq moderate_pa_unit, and vigorous_pa_unit as bytes object 

unit_map = {
    b"D": "Day",
    b"W": "Week",
    b"M": "Month",
    b"Y": "Year",
    b"": np.nan
}

paq_req_columns["moderate_pa_unit"] = (
    paq_req_columns["moderate_pa_unit"].map(unit_map)
)

paq_req_columns["vigorous_pa_unit"] = (
    paq_req_columns["vigorous_pa_unit"].map(unit_map)
)

In [136]:
# checking for missingness and duplicates

datasets = {
    "demo": demo_cohort,
    "bmx": bmx_req_columns,
    "paq": paq_req_columns,
    "smq": smq_req_columns,
    "bpxo": bpxo_req_columns,
    "bpq": bpq_req_columns,
    "diq": diq_req_columns,
    "ghb": ghb_req_columns,
    "glu": glu_req_columns
}

for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print("Shape:", df.shape)
    print("Unique participants:", df["participant_id"].nunique())
    print("Duplicate IDs:", df["participant_id"].duplicated().sum())
    print("Missing IDs:", df["participant_id"].isna().sum())
    print(df.isna().sum())


DEMO
Shape: (7371, 9)
Unique participants: 7371
Duplicate IDs: 0
Missing IDs: 0
participant_id             0
age_years                  0
sex                        0
race_ethnicity             0
education_level           21
income_poverty_ratio    1210
mec_exam_weight            0
survey_stratum             0
survey_psu                 0
dtype: int64

BMX
Shape: (8860, 3)
Unique participants: 8860
Duplicate IDs: 0
Missing IDs: 0
participant_id      0
bmi               389
waist_cm          670
dtype: int64

PAQ
Shape: (8153, 8)
Unique participants: 8153
Duplicate IDs: 0
Missing IDs: 0
participant_id                    0
moderate_pa_frequency            18
moderate_pa_unit               1763
moderate_pa_minutes_session    1763
vigorous_pa_frequency            14
vigorous_pa_unit               4467
vigorous_pa_minutes_session    4466
sedentary_min_day                15
dtype: int64

SMQ
Shape: (9015, 3)
Unique participants: 9015
Duplicate IDs: 0
Missing IDs: 0
participant_id           

In [137]:
# evaluating missingness percentage

for name, df in datasets.items():
    print(f"\n{name.upper()}")

    missing = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percent": (df.isna().mean() * 100).round(2)
    })

    print(missing)


DEMO
                      missing_count  missing_percent
participant_id                    0             0.00
age_years                         0             0.00
sex                               0             0.00
race_ethnicity                    0             0.00
education_level                  21             0.28
income_poverty_ratio           1210            16.42
mec_exam_weight                   0             0.00
survey_stratum                    0             0.00
survey_psu                        0             0.00

BMX
                missing_count  missing_percent
participant_id              0             0.00
bmi                       389             4.39
waist_cm                  670             7.56

PAQ
                             missing_count  missing_percent
participant_id                           0             0.00
moderate_pa_frequency                   18             0.22
moderate_pa_unit                      1763            21.62
moderate_pa_minutes_sessio

## Cohort level missingness
Among the people who appear in the cohort, how many do not have this variable available?

Merging on participant_id to understand missigness in case of a merge.

In [138]:
cohort_datasets = {
    "demo": demo_cohort
}

for name, df in datasets.items():

    if name == "demo":
        continue

    cohort_datasets[name] = (
        demo_cohort[["participant_id"]]
        .merge(
            df,
            on="participant_id",
            how="left",
            validate="one_to_one"
        )
    )

In [139]:
for name, df in cohort_datasets.items():

    print(f"\n{'=' * 50}")
    print(name.upper())
    print("=" * 50)

    missing = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percent": (df.isna().mean() * 100).round(2)
    })

    print(missing)


DEMO
                      missing_count  missing_percent
participant_id                    0             0.00
age_years                         0             0.00
sex                               0             0.00
race_ethnicity                    0             0.00
education_level                  21             0.28
income_poverty_ratio           1210            16.42
mec_exam_weight                   0             0.00
survey_stratum                    0             0.00
survey_psu                        0             0.00

BMX
                missing_count  missing_percent
participant_id              0             0.00
bmi                      1463            19.85
waist_cm                 1665            22.59

PAQ
                             missing_count  missing_percent
participant_id                           0             0.00
moderate_pa_frequency                    8             0.11
moderate_pa_unit                      1623            22.02
moderate_pa_minutes_sessio

## Understanding categorical value distributions

In [140]:
categorical_columns = {
    "demo": [
        "sex",
        "race_ethnicity",
        "education_level"
    ],

    "paq": [
        "moderate_pa_unit",
        "vigorous_pa_unit"
    ],

    "smq": [
        "ever_100_cigarettes",
        "current_smoking_frequency"
    ],

    "bpq": [
        "hypertension_history",
        "bp_medication",
        "high_cholesterol_history",
        "cholesterol_medication"
    ],

    "diq": [
        "diagnosed_diabetes",
        "prediabetes_history"
    ]
}

for name, columns in categorical_columns.items():

    print(f"\n{'=' * 40}")
    print(name.upper())
    print("=" * 40)

    for column in columns:
        print(f"\n")
        print(datasets[name][column].value_counts(dropna=False))


DEMO


sex
2.0    3883
1.0    3488
Name: count, dtype: int64


race_ethnicity
3.0    4360
4.0     902
2.0     728
1.0     500
7.0     475
6.0     406
Name: count, dtype: int64


education_level
5.0    2488
4.0    2221
3.0    1652
2.0     626
1.0     363
NaN      21
Name: count, dtype: int64

PAQ


moderate_pa_unit
Week     4851
NaN      1763
Day       985
Month     472
Year       82
Name: count, dtype: int64


vigorous_pa_unit
NaN      4467
Week     2590
Month     711
Day       235
Year      150
Name: count, dtype: int64

SMQ


ever_100_cigarettes
2.0    4878
1.0    3243
NaN     880
9.0       7
7.0       7
Name: count, dtype: int64


current_smoking_frequency
NaN    5772
3.0    2053
1.0     952
2.0     238
Name: count, dtype: int64

BPQ


hypertension_history
2.0    5518
1.0    2969
9.0      10
NaN       3
7.0       1
Name: count, dtype: int64


bp_medication
NaN    5532
1.0    2442
2.0     523
9.0       4
Name: count, dtype: int64


high_cholesterol_history
2.0    5348
1.0    3096
9.

## Audit NHANES special response codes

In [141]:
special_codes = {

    "paq": {
        "moderate_pa_frequency": [7777, 9999],
        "moderate_pa_minutes_session": [7777, 9999],
        "vigorous_pa_frequency": [7777, 9999],
        "vigorous_pa_minutes_session": [7777, 9999],
        "sedentary_min_day": [7777, 9999]
    },

    "smq": {
        "ever_100_cigarettes": [7, 9]
    },

    "bpq": {
        "hypertension_history": [7, 9],
        "bp_medication": [7, 9],
        "high_cholesterol_history": [7, 9],
        "cholesterol_medication": [7, 9]
    },

    "diq": {
        "diagnosed_diabetes": [7, 9],
        "prediabetes_history": [7, 9]
    }
}

special_code_findings = []

for dataset_name, columns in special_codes.items():

    df = cohort_datasets[dataset_name]

    for column, codes in columns.items():

        for code in codes:

            count = (df[column] == code).sum()

            if count > 0:
                special_code_findings.append({
                    "dataset": dataset_name.upper(),
                    "column": column,
                    "code": code,
                    "count": count
                })

special_code_findings = pd.DataFrame(special_code_findings)

special_code_findings

,dataset,column,code,count
0,PAQ,moderate_pa_frequency,7777,7
1,PAQ,moderate_pa_frequency,9999,37
2,PAQ,moderate_pa_minutes_session,9999,18
3,PAQ,vigorous_pa_frequency,7777,2
4,PAQ,vigorous_pa_frequency,9999,35
5,PAQ,vigorous_pa_minutes_session,7777,1
6,PAQ,vigorous_pa_minutes_session,9999,10
7,PAQ,sedentary_min_day,7777,5
8,PAQ,sedentary_min_day,9999,66
9,SMQ,ever_100_cigarettes,7,7


## Audit continuous values

In [142]:
numeric_columns = {

    "demo": [
        "age_years",
        "income_poverty_ratio"
    ],

    "bmx": [
        "bmi",
        "waist_cm"
    ],

    "paq": [
        "moderate_pa_frequency",
        "moderate_pa_minutes_session",
        "vigorous_pa_frequency",
        "vigorous_pa_minutes_session",
        "sedentary_min_day"
    ],

    "bpxo": [
        "systolic_1",
        "systolic_2",
        "systolic_3",
        "diastolic_1",
        "diastolic_2",
        "diastolic_3"
    ],

    "ghb": [
        "hba1c_pct"
    ],

    "glu": [
        "fasting_glucose_mg_dl"
    ]
}

In [143]:
for name, columns in numeric_columns.items():

    print(f"\n{'=' * 50}")
    print(name.upper())
    print("=" * 50)

    display(
        cohort_datasets[name][columns]
        .describe(
            percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
        )
        .T
    )


DEMO


,count,mean,std,min,1%,5%,50%,95%,99%,max
age_years,7371.0,54.889160,17.161332,20.0,21.0,25.00,58.00,80.0,80.0,80.0
income_poverty_ratio,6161.0,2.921417,1.655707,0.0,0.0,0.46,2.83,5.0,5.0,5.0



BMX


,count,mean,std,min,1%,5%,50%,95%,99%,max
bmi,5908.0,29.813761,7.329927,11.1,18.2,20.4,28.5,43.6,52.693,74.8
waist_cm,5706.0,101.080775,16.868652,62.4,70.2,76.4,99.5,131.0,148.190,187.0



PAQ


...

,count,mean,std,min,1%,5%,50%,95%,99%,max
systolic_1,5804.0,123.210544,18.539453,75.0,88.03,98.0,121.0,157.0,180.00,232.0
systolic_2,5797.0,123.065551,18.480937,65.0,88.00,98.0,121.0,157.0,178.00,233.0
systolic_3,5785.0,122.868280,18.375472,62.0,88.00,98.0,121.0,156.0,179.16,232.0
diastolic_1,5804.0,75.355272,11.360566,37.0,52.00,59.0,75.0,95.0,107.00,142.0
diastolic_2,5797.0,74.678972,11.286436,32.0,50.00,58.0,74.0,95.0,106.04,139.0
diastolic_3,5785.0,74.309248,11.193505,26.0,50.00,58.0,74.0,94.0,104.16,136.0



GHB


,count,mean,std,min,1%,5%,50%,95%,99%,max
hba1c_pct,5709.0,5.794377,1.115476,3.2,4.5,4.9,5.5,7.7,10.992,17.1



GLU


,count,mean,std,min,1%,5%,50%,95%,99%,max
fasting_glucose_mg_dl,3174.0,109.983302,34.29373,59.0,78.73,86.0,101.0,165.35,277.27,561.0


## Audit weight values

mec_exam_weight
phlebotomy_weight
fasting_subsample_weight
survey_stratum
survey_psu

In [144]:
weight_columns = {
    "demo": "mec_exam_weight",
    "ghb": "phlebotomy_weight",
    "glu": "fasting_subsample_weight"
}

for name, column in weight_columns.items():

    df = cohort_datasets[name]

    print(
        name.upper(),
        column,
        "zero:",
        (df[column] == 0).sum(),
        "missing:",
        df[column].isna().sum()
    )

DEMO mec_exam_weight zero: 1376 missing: 0
GHB phlebotomy_weight zero: 0 missing: 1376
GLU fasting_subsample_weight zero: 0 missing: 3989
